# N097 · 字典序构造与剩余可行性

**目标：** 兼顾当前更优字符和未来完成约束。

**先修：** N054, N020, N093。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 单调栈；剩余次数；去重标记；前导零；构造可行性。

## 从问题到算法

字典序目标依赖最早出现差异的位置。删除k位时，当前更小字符可以替换左侧较大字符，但必须仍有删除预算。去重字母还要保证被弹出的字母以后再次出现。重排字符串不是字典序最小问题：用最大频次堆并暂存上次字符，禁止立即重复。

## 最小实现

**本章接口：** `remove_k_digits(num, k)`、`remove_duplicate_letters(s)`、`reorganize_string(s)`

In [1]:
from collections import Counter
from heapq import heapify,heappush,heappop

def remove_k_digits(num,k):
    stack=[]
    for c in num:
        while k and stack and stack[-1]>c: stack.pop(); k-=1
        stack.append(c)
    if k: del stack[-k:]
    return ''.join(stack).lstrip('0') or '0'

def remove_duplicate_letters(s):
    last={c:i for i,c in enumerate(s)}; used=set(); stack=[]
    for i,c in enumerate(s):
        if c in used: continue
        while stack and stack[-1]>c and last[stack[-1]]>i: used.remove(stack.pop())
        stack.append(c); used.add(c)
    return ''.join(stack)

def reorganize_string(s):
    counts=Counter(s)
    if max(counts.values(),default=0)>(len(s)+1)//2: return ''
    heap=[(-n,c) for c,n in counts.items()]; heapify(heap); pending=(0,''); out=[]
    while heap:
        neg,c=heappop(heap); out.append(c)
        if pending[0]<0: heappush(heap,pending)
        pending=(neg+1,c)
    return ''.join(out) if pending[0]==0 else ''

## 正确性、前提与复杂度

单调栈删除发生在能改善最早差异的位置，未用完预算时应删末尾。去重栈的弹出必须有未来副本，才能维持覆盖所有字母的可行性。重排的必要条件是最大频数不超过其余字符数加1；优先安排最多的可选字符并隔离上一字符，可维持该可行性条件。

**代价：** 删除数字与去重字母O(n)时间、O(n)空间；重排O(n log σ)时间、O(σ+n)空间，σ为不同字符数。无可行重排返回空串；空输入也返回空串。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='cbacdcbc'; last={c:i for i,c in enumerate(s)}; stack=[]; used=set(); rows=[]
for i,c in enumerate(s):
    if c not in used:
        while stack and stack[-1]>c and last[stack[-1]]>i: used.remove(stack.pop())
        stack.append(c); used.add(c)
    rows.append((i,c,''.join(stack)))
show_table(['位置','读入','栈'],rows)

位置,读入,栈
0,c,c
1,b,b
2,a,a
3,c,ac
4,d,acd
5,c,acd
6,b,acdb
7,c,acdb


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert remove_k_digits('1432219',3)=='1219'
assert remove_k_digits('10200',1)=='200'
assert remove_duplicate_letters('cbacdcbc')=='acdb'
from itertools import product,combinations
for n in range(1,7):
    for chars in product('ab',repeat=n):
        s=''.join(chars); answer=reorganize_string(s)
        feasible=max(Counter(s).values())<=(n+1)//2
        assert bool(answer)==feasible
        if feasible: assert Counter(answer)==Counter(s) and all(a!=b for a,b in zip(answer,answer[1:]))
for chars in product('abc',repeat=5):
    s=''.join(chars); k=len(set(s)); options=[''.join(s[i] for i in ids) for ids in combinations(range(5),k) if len({s[i] for i in ids})==k]
    assert remove_duplicate_letters(s)==min(options)
for chars in product('012',repeat=4):
    s=''.join(chars)
    for k in range(5):
        choices=[int(''.join(s[i] for i in ids) or '0') for ids in combinations(range(4),4-k)]
        assert int(remove_k_digits(s,k))==min(choices)
print('N097: 所有本章断言通过')

N097: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 证明字符以后不再出现时不能弹出。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较数值最小与字典序最小。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 402. Remove K Digits（canonical）
- 316. Remove Duplicate Letters（canonical）
- 767. Reorganize String（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。